# Knowledge Graph RAG Implementation

This notebook demonstrates building a Knowledge Graph RAG system.

## Setup

In [ ]:
!pip install langchain langchain-community langchain-openai neo4j networkx

In [ ]:
import os
os.environ["NEO4J_PASSWORD"] = "your-neo4j-password"

## Create Sample Data with Relationships

In [ ]:
from langchain_core.documents import Document

sample_docs = [
    Document(
        page_content="Apple was founded in 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne.",
        metadata={"source": "doc1", "type": "company"}
    ),
    Document(
        page_content="Apple's headquarters is located in Cupertino, California.",
        metadata={"source": "doc2", "type": "location"}
    ),
    Document(
        page_content="Steve Jobs co-founded Apple and later founded NeXT and Pixar.",
        metadata={"source": "doc3", "type": "person"}
    ),
    Document(
        page_content="Tim Cook became CEO of Apple in 2011, succeeding Steve Jobs.",
        metadata={"source": "doc4", "type": "person"}
    ),
    Document(
        page_content="Apple produces iPhone, Mac, iPad, and Apple Watch.",
        metadata={"source": "doc5", "type": "product"}
    ),
]

print(f"Created {len(sample_docs)} sample documents")

## Build Knowledge Graph

In [ ]:
from langchain_community.graphs import Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_ollama import ChatOllama

# Connect to Neo4j (or use memory graph for demo)
try:
    graph = Neo4jGraph(
        url="bolt://localhost:7687",
        username="neo4j",
        password=os.environ["NEO4J_PASSWORD"]
    )
    print("Connected to Neo4j")
except:
    print("Neo4j not available, using in-memory graph")

## Extract Knowledge Graph with LLM

In [ ]:
llm = ChatOllama(model="llama3.2")

graph_transformer = LLMGraphTransformer(llm=llm)

# Convert documents to graph
graph_documents = graph_transformer.convert_to_graph_documents(sample_docs)

print(f"Created {len(graph_documents)} graph documents")

# Display first graph document
if graph_documents:
    print("\nNodes:", [n.id for n in graph_documents[0].nodes])
    print("Relationships:", [(r.source.id, r.type, r.target.id) for r in graph_documents[0].relationships])

## Hybrid Retrieval Implementation

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

# Create vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_documents(sample_docs, embeddings)

print("Vector store created")

In [ ]:
class HybridKGRetriever:
    """Combines vector and graph retrieval."""
    
    def __init__(self, vectorstore, llm):
        self.vectorstore = vectorstore
        self.llm = llm
    
    def retrieve(self, query: str, k: int = 4):
        # Vector retrieval
        vector_results = self.vectorstore.similarity_search(query, k=k)
        
        # Extract entities and query graph
        entities = self._extract_entities(query)
        graph_results = self._graph_retrieve(entities)
        
        # Combine and deduplicate
        combined = vector_results + graph_results
        return combined[:k]
    
    def _extract_entities(self, query: str) -> list:
        prompt = f"Extract key entities from: {query}"
        result = self.llm.invoke(prompt)
        return [e.strip() for e in result.split(",")]
    
    def _graph_retrieve(self, entities: list):
        # Simulated graph retrieval
        return []

retriever = HybridKGRetriever(vectorstore, llm)
print("Hybrid retriever ready")

## Query the KG-RAG System

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

# Query
query = "Who founded Apple and what products do they make?"
result = qa_chain.invoke({"query": query})

print(f"Question: {query}")
print(f"\nAnswer: {result['result']}")

## Next Steps
1. Set up Neo4j for persistent storage
2. Add more complex relationships
3. Implement graph traversal queries
4. Add entity resolution